# 3. Download raw videos

Download the first `VIDEO_LIMIT` clips listed in the raw video manifest. Each file is downloaded to a `.part` file and only renamed when complete.

**Rerun behaviour** is controlled by `OVERWRITE`:

- `OVERWRITE = False` (default): existing non-empty AVI files are skipped; only missing (or interrupted) downloads are fetched. `video_downloads.csv` keeps its earlier rows and is updated for the videos in this run.
- `OVERWRITE = True`: every selected video is downloaded again and `video_downloads.csv` is rewritten.

A video that the S3 bucket does not have (HTTP 404) or that fails mid-transfer does not stop the run: its `.part` file is removed, the row is logged as `missing_remote` or `download_failed` with the reason in the new `detail` column, and the loop continues with the next video. Rerunning with `OVERWRITE = False` retries those videos, since no AVI was written for them.

When run from `00_run_annotation_pipeline.ipynb`, the runner's `OVERWRITE` takes precedence.

In [3]:
import csv
import os
from collections import Counter
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import urlopen

# True: re-download every selected video even if the AVI already exists.
# False: only download videos that are missing (or were left incomplete as .part files).
OVERWRITE = False
OVERWRITE = os.environ.get("PIPELINE_OVERWRITE", str(OVERWRITE)).strip().lower() in {"1", "true", "yes"}

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
VIDEO_DIRECTORY = DATA_DIR / "videos"
S3_BUCKET_URL = "https://mevadata-public-01.s3.amazonaws.com"
S3_PREFIX = "drops-123-r13"
VIDEO_LIMIT = 80
CHUNK_SIZE = 1024 * 1024

with (DATA_DIR / "video_names.csv").open(newline="", encoding="utf-8") as input_file:
    video_names = [row["video_name"] for row in csv.DictReader(input_file)][:VIDEO_LIMIT]

print("=" * 72, flush=True)
print(f"Stage 3: downloading {len(video_names)} videos (OVERWRITE={OVERWRITE})", flush=True)
print(f"Destination: {VIDEO_DIRECTORY}", flush=True)
print("=" * 72, flush=True)

VIDEO_DIRECTORY.mkdir(parents=True, exist_ok=True)
downloads = []
for index, video_name in enumerate(video_names, start=1):
    print(f"[{index}/{len(video_names)}] {video_name}", flush=True)
    date, start_time, *_ = video_name.split(".")
    url = f"{S3_BUCKET_URL}/{quote(f'{S3_PREFIX}/{date}/{start_time[:2]}/{video_name}.r13.avi')}"
    destination = VIDEO_DIRECTORY / f"{video_name}.r13.avi"
    temporary_destination = destination.with_suffix(".avi.part")

    if not OVERWRITE and destination.is_file() and destination.stat().st_size > 0:
        status, detail = "skipped_existing", ""
    else:
        temporary_destination.unlink(missing_ok=True)
        try:
            with urlopen(url, timeout=60) as response, temporary_destination.open("wb") as output:
                while chunk := response.read(CHUNK_SIZE):
                    output.write(chunk)
        except HTTPError as error:
            # A clip listed in the manifest may not exist in the bucket (404). Log it and keep going.
            temporary_destination.unlink(missing_ok=True)
            status = "missing_remote" if error.code == 404 else "download_failed"
            detail = f"HTTP {error.code} {error.reason}"
        except OSError as error:
            # Network drop, timeout or disk error: leave no partial file behind and move to the next video.
            temporary_destination.unlink(missing_ok=True)
            status, detail = "download_failed", f"{type(error).__name__}: {error}"
        else:
            temporary_destination.replace(destination)
            status, detail = "downloaded", ""
    downloads.append(
        {
            "video_name": video_name,
            "s3_url": url,
            "local_path": str(destination),
            "status": status,
            "detail": detail,
        }
    )
    print(f"  {status}: {destination.name}{f' ({detail})' if detail else ''}", flush=True)

# Without OVERWRITE, keep log rows for videos outside this run and update the ones processed now.
downloads_log = DATA_DIR / "video_downloads.csv"
log_rows = {}
if not OVERWRITE and downloads_log.is_file():
    with downloads_log.open(newline="", encoding="utf-8") as input_file:
        log_rows = {row["video_name"]: row for row in csv.DictReader(input_file)}
log_rows.update({row["video_name"]: row for row in downloads})
with downloads_log.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=["video_name", "s3_url", "local_path", "status", "detail"])
    writer.writeheader()
    writer.writerows(log_rows.values())

counts = Counter(row["status"] for row in downloads)
print(
    f"OVERWRITE={OVERWRITE}: {counts['downloaded']} downloaded, "
    f"{counts['skipped_existing']} already present, "
    f"{counts['missing_remote']} not in the bucket, "
    f"{counts['download_failed']} failed."
)
for row in downloads:
    if row["status"] in {"missing_remote", "download_failed"}:
        print(f"  {row['status']}: {row['video_name']} ({row['detail']})")

Stage 3: downloading 80 videos (OVERWRITE=False)
Destination: /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw/videos
[1/80] 2018-03-05.13-15-00.13-20-00.bus.G340
  skipped_existing: 2018-03-05.13-15-00.13-20-00.bus.G340.r13.avi
[2/80] 2018-03-05.13-15-00.13-20-00.bus.G505
  skipped_existing: 2018-03-05.13-15-00.13-20-00.bus.G505.r13.avi
[3/80] 2018-03-05.13-15-00.13-20-00.bus.G506
  skipped_existing: 2018-03-05.13-15-00.13-20-00.bus.G506.r13.avi
[4/80] 2018-03-05.13-15-00.13-20-00.hospital.G341
  skipped_existing: 2018-03-05.13-15-00.13-20-00.hospital.G341.r13.avi
[5/80] 2018-03-05.13-15-00.13-20-00.school.G423
  skipped_existing: 2018-03-05.13-15-00.13-20-00.school.G423.r13.avi
[6/80] 2018-03-05.13-15-00.13-20-00.school.G424
  skipped_existing: 2018-03-05.13-15-00.13-20-00.school.G424.r13.avi
[7/80] 2018-03-05.13-15-01.13-20-01.school.G328
  skipped_existing: 2018-03-05.13-15-01.13-20-01.school.G328.r13.avi
[8/80] 2018-03-05.13-20-00.13-25-00.bus.G340
  skippe